# b4-qlora｜4-bit QLoRA 微調

**目標**：用 4-bit 量化（QLoRA）在 T4 GPU 上微調 Llama-3.2-3B-Instruct

**與 b3-lora 的差異**：
| | b3-lora | b4-qlora |
|---|---|---|
| 模型精度 | bfloat16（全精度） | 4-bit NF4（量化） |
| 記憶體 | ~6GB（MPS） | ~3GB（VRAM） |
| 硬體 | Mac MPS | T4 GPU |
| 額外步驟 | 無 | `prepare_model_for_kbit_training()` |

**訓練規模**：1 epoch，1000 筆（約 2% 訓練集）
全量訓練實測約 16 小時，遠超 Colab 免費配額；b4 目的是跑通 QLoRA 流程，不需全量資料
實際免費配額約 2.5 小時（因帳號用量而異），5000 筆估算仍超時，最終以 1000 筆完成

**執行前確認**：
- Runtime → Change runtime type → **T4 GPU**
- 左側 🔑 Secrets 已設定 `HF_TOKEN`、`WANDB_API_KEY`（需 40 字元以上）
- Google Drive 已有 `MyDrive/Tangram/data/dataset/`

In [ ]:
# ── 確認 GPU ─────────────────────────────────────────────
import torch

assert torch.cuda.is_available(), "GPU 未啟用，請到 Runtime → Change runtime type → T4 GPU"
print(f"GPU：{torch.cuda.get_device_name(0)}")
print(f"VRAM：{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── 安裝套件 ──────────────────────────────────────────────
# bitsandbytes 是 4-bit 量化的核心，只能在 CUDA 環境執行（不支援 MPS）
!pip install -q transformers datasets trl peft bitsandbytes accelerate

In [ ]:
# ── 掛載 Google Drive + 讀取 Secrets ─────────────────────
from google.colab import drive, userdata

drive.mount('/content/drive')

HF_TOKEN = userdata.get('HF_TOKEN')
print("HF_TOKEN：", "✓ 已載入" if HF_TOKEN else "✗ 未設定，請到左側 🔑 新增 HF_TOKEN")

In [ ]:
# ── WandB 登入 ────────────────────────────────────────────
import os
import wandb
from google.colab import userdata

WANDB_API_KEY = userdata.get('WANDB_API_KEY')
if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)
    os.environ["WANDB_PROJECT"] = "tangram"
    print("WandB：✓ 已登入，project = tangram")
else:
    print("WandB：✗ 未設定 WANDB_API_KEY，訓練 log 將不會上傳")

In [ ]:
# ── 設定路徑與超參數 ──────────────────────────────────────
from pathlib import Path

MODEL_ID       = "meta-llama/Llama-3.2-3B-Instruct"
DRIVE_BASE     = Path("/content/drive/MyDrive/Tangram")
DATASET_PATH   = DRIVE_BASE / "data/dataset"
CHECKPOINT_DIR = DRIVE_BASE / "checkpoints/b4"
OUTPUT_LOG     = DRIVE_BASE / "outputs/b4_qlora_log.json"
MAX_SEQ_LENGTH = 1024
ALPACA_RATIO   = 0.05

(CHECKPOINT_DIR / "adapter").mkdir(parents=True, exist_ok=True)
OUTPUT_LOG.parent.mkdir(parents=True, exist_ok=True)

print(f"資料集路徑：{DATASET_PATH}")
print(f"資料集存在：{'✓' if DATASET_PATH.exists() else '✗ 請確認已上傳到 Drive'}")

In [ ]:
# ── 載入 tokenizer ────────────────────────────────────────
from transformers import AutoTokenizer

print("載入 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = MAX_SEQ_LENGTH
print("完成")

In [ ]:
# ── 載入資料集 + 混入 5% Alpaca ───────────────────────────
from datasets import load_from_disk, Dataset, concatenate_datasets, load_dataset

print("載入 Tangram 資料集...")
dataset  = load_from_disk(str(DATASET_PATH))
train_ds = dataset["train"].select_columns(["text"])
val_ds   = dataset["validation"].select_columns(["text"])
print(f"  訓練集：{len(train_ds)} 筆，驗證集：{len(val_ds)} 筆")

def format_alpaca(sample):
    user_content = sample["instruction"]
    if sample.get("input"):
        user_content += f"\n{sample['input']}"
    messages = [
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": sample["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

alpaca_n   = int(len(train_ds) * ALPACA_RATIO)
print(f"載入 Alpaca（取 {alpaca_n} 筆）...")
alpaca_raw = load_dataset("yahma/alpaca-cleaned", split=f"train[:{alpaca_n}]")
alpaca_ds  = Dataset.from_list([format_alpaca(s) for s in alpaca_raw])
train_mixed = concatenate_datasets([train_ds, alpaca_ds]).shuffle(seed=42)
print(f"  混合後訓練集：{len(train_mixed)} 筆")

In [ ]:
# ── 載入模型（4-bit 量化）+ 套上 LoRA adapter ────────────
#
# BitsAndBytesConfig：
#   load_in_4bit          → 把模型權重壓縮成 4-bit 儲存
#   bnb_4bit_quant_type   → NF4 是 QLoRA 論文推薦的量化格式
#   bnb_4bit_compute_dtype→ 計算梯度時升回 float16
#   bnb_4bit_use_double_quant → 對量化常數再量化，多省約 0.4 bits/param
#
# prepare_model_for_kbit_training()：
#   1. 凍結量化層的 weight
#   2. 把 LayerNorm 升回 float32（避免梯度下溢）
#   3. 讓 embedding layer 可計算梯度
#   這步是 4-bit 模型特有的，b3-lora 不需要
#
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("載入模型（4-bit 量化）...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
)
print(f"  VRAM 使用：{torch.cuda.memory_allocated() / 1e9:.2f} GB")

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # 維持與 b3 一致以利 b5 比較
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
ratio     = trainable / total * 100
print(f"  全部參數：  {total / 1e9:.3f}B")
print(f"  可訓練參數：{trainable / 1e6:.2f}M ({ratio:.4f}%)")
assert ratio < 1.0, f"可訓練參數比例 {ratio:.4f}% 超過 1%"
print("  DoD 確認：可訓練參數 < 1% ✓")

In [ ]:
# ── Smoke Test（已驗證通過，略過）────────────────────────
# 如需重新驗證鏈路，取消下方註解跑 50 steps
#
# from trl import SFTTrainer, SFTConfig
# smoke_args = SFTConfig(
#     output_dir="/content/smoke_test",
#     max_steps=50,
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=4,
#     gradient_checkpointing=True,
#     gradient_checkpointing_kwargs={"use_reentrant": False},
#     learning_rate=2e-4,
#     dataset_text_field="text",
#     logging_steps=10,
#     eval_strategy="no",
#     save_strategy="no",
#     seed=42,
#     report_to="none",
# )
# smoke_trainer = SFTTrainer(
#     model=model,
#     args=smoke_args,
#     train_dataset=train_mixed.select(range(200)),
#     processing_class=tokenizer,
# )
# smoke_trainer.train()
# print("Smoke test 通過 ✓")

print("Smoke test 已略過（鏈路已驗證）")

In [ ]:
# ── 正式訓練 ──────────────────────────────────────────────
# 使用 1000 筆（約 2% 訓練集）
# 全量約 16 小時；5000 筆在 2.5 小時配額下也超時，實測以 1000 筆完成
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=2e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    dataset_text_field="text",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=42,
    report_to="wandb" if WANDB_API_KEY else "none",
    run_name="tangram-b4-qlora",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_mixed.select(range(1000)),
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

print("開始訓練（1 epoch，1000 筆，預計約 1.5 小時）...")
train_result = trainer.train()
print("\n訓練完成")
print(train_result.metrics)

In [ ]:
# ── 儲存 adapter + log ────────────────────────────────────
import json

adapter_dir = CHECKPOINT_DIR / "adapter"
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"LoRA adapter 已存至 {adapter_dir}")

log = {
    "lora_r": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "target_modules": list(lora_config.target_modules),
    "quantization": "4-bit NF4 + double quant",
    "train_samples": 1000,
    "trainable_params": trainable,
    "trainable_ratio_pct": round(ratio, 4),
    "train_runtime_sec": train_result.metrics.get("train_runtime"),
    "train_loss": train_result.metrics.get("train_loss"),
    "history": trainer.state.log_history,
}
with open(OUTPUT_LOG, "w", encoding="utf-8") as f:
    json.dump(log, f, ensure_ascii=False, indent=2)
print(f"訓練 log 已存至 {OUTPUT_LOG}")

print("\n" + "=" * 50)
print("DoD 驗證")
print("=" * 50)
print(f"  可訓練參數 < 1%  ✓ ({ratio:.4f}%)")
train_losses = [e["loss"] for e in trainer.state.log_history if "loss" in e]
eval_losses  = [e["eval_loss"] for e in trainer.state.log_history if "eval_loss" in e]
print(f"  final train_loss = {train_losses[-1]:.4f}" if train_losses else "  train_loss: 無資料")
print(f"  final eval_loss  = {eval_losses[-1]:.4f}"  if eval_losses  else "  eval_loss: 無資料")

In [ ]:
# ── 下載 log 到本機 ───────────────────────────────────────
# 執行這個 cell 會觸發瀏覽器下載
# 下載後放到本機的 outputs/b4_qlora_log.json
from google.colab import files
files.download(str(OUTPUT_LOG))